# Safe validation for `validate_tar_gz` 

When using a direct validation function, such as `validate_tar_gz`, any exceptions raised during validation are passed to the calling function. This means the caller is responsible for handling validation errors.

However, there are situations where you may prefer a simple `True` or `False` result instead of handling exceptions yourself. You might also want to catch the exception and integrate it into your own logging or error-reporting functionality.

After all, the reason why validation failed is often important information that should not be lost.

Below is an example of how to implement a `safe_validate_tar_gz` wrapper around `validate_tar_gz`. It accepts the same validation parameters, but catches exceptions and returns `False` when validation fails.

This notebook demonstrates direct TAR.GZ validation and a safe wrapper that returns `True` or `False` while logging validation failures.

The notebook creates its own temporary archive, smaller than 100 KB. No external archive is required.

## 1. Install Python File Audit

```bash
pip install fileaudit
```

## 2. Imports and logging

In [1]:
import logging
import tarfile
import tempfile
from pathlib import Path

from fileaudit.targz_check import validate_tar_gz, TarValidationError

logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s: %(message)s',
)
logger = logging.getLogger('tar_gz_demo')

## 3. Define `safe_validate_tar_gz`

The direct API returns `True` when validation succeeds and raises an exception when validation fails. This wrapper catches validation exceptions, logs the reason, and returns `False`.

In [2]:
def safe_validate_tar_gz(directory, **kwargs):
    """Validate a TAR.GZ archive and return a Boolean result.

    Args:
        directory: Path to the TAR.GZ archive.
        **kwargs: Additional keyword arguments passed to validate_tar_gz().

    Returns:
        True if validation succeeds; False if validation raises an exception.
    """
    try:
        return validate_tar_gz(directory, **kwargs)
    except Exception:
        logger.exception('TAR.GZ validation failed for %s', directory)
        return False

## 4. Create a temporary TAR.GZ archive

The archive contains a tiny text file. The cell checks that the resulting compressed archive is smaller than 100 KB.

In [3]:
temp_dir = tempfile.TemporaryDirectory()
work_dir = Path(temp_dir.name)
source_file = work_dir / 'hello.txt'
archive_path = work_dir / 'demo.tar.gz'

source_file.write_text(
    'Hello! This is a Python File Audit validation demo.\\n',
    encoding='utf-8',
)

with tarfile.open(archive_path, mode='w:gz') as tar:
    tar.add(source_file, arcname='hello.txt')

archive_size = archive_path.stat().st_size
assert archive_size < 100 * 1024, f'Archive too large: {archive_size} bytes'

print(f'Created: {archive_path}')
print(f'Archive size: {archive_size:,} bytes (< 100 KB)')

Created: /tmp/tmpuesnmlxs/demo.tar.gz
Archive size: 250 bytes (< 100 KB)


## 5. Direct validation with `validate_tar_gz`

A valid archive should return `True`. If a check fails, the direct function raises an exception that the caller must handle.

In [4]:
try:
    result = validate_tar_gz(archive_path)
    print(f'Direct validation result: {result}')
except TarValidationError as exc:
    print(f'Direct validation raised TarValidationError: {exc}')

Direct validation result: True


## 6. Demonstrate a validation failure

This example deliberately sets `max_tar_members=0`. The archive contains one member, so it should fail this limit. The keyword argument is passed through to `validate_tar_gz`.

In [5]:
try:
    validate_tar_gz(archive_path, max_tar_members=0)
except TarValidationError as exc:
    print(f'Caught validation exception: {exc}')

Caught validation exception: FileAudit Security Validation Failed - TAR validation failed: FileAudit Security Validation Failed - Archive contains 1 members, exceeding maximum of 0


## 7. Safe validation

The same successful and failing checks using the wrapper. A failure is logged with its traceback and represented by `False`.

In [6]:
valid_result = safe_validate_tar_gz(archive_path)
print(f'Safe validation of valid archive: {valid_result}')

invalid_result = safe_validate_tar_gz(archive_path, max_tar_members=0)
print(f'Safe validation with failing limit: {invalid_result}')

ERROR: TAR.GZ validation failed for /tmp/tmpuesnmlxs/demo.tar.gz
Traceback (most recent call last):
  File "/home/maikel/projects/pythondev/fileaudit/src/fileaudit/targz_check.py", line 381, in _validate_tar_gz_file
    raise TarValidationError(
    ...<2 lines>...
    )
fileaudit.targz_check.TarValidationError: FileAudit Security Validation Failed - Archive contains 1 members, exceeding maximum of 0

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/tmp/ipykernel_24391/3408463422.py", line 12, in safe_validate_tar_gz
    return validate_tar_gz(directory, **kwargs)
  File "/home/maikel/projects/pythondev/fileaudit/src/fileaudit/targz_check.py", line 241, in validate_tar_gz
    _validate_tar_gz_file(
    ~~~~~~~~~~~~~~~~~~~~~^
        func_or_path,
        ^^^^^^^^^^^^^
    ...<6 lines>...
        resolved_depth,
        ^^^^^^^^^^^^^^^
    )
    ^
  File "/home/maikel/projects/pythondev/fileaudit/src/fileaudit/targz_check.

Safe validation of valid archive: True
Safe validation with failing limit: False


## 8. Use the Boolean result in application code

This is useful in batch jobs or workflows where the caller wants to branch on success or failure instead of handling an exception at every call site.

In [7]:
if safe_validate_tar_gz(archive_path):
    print('Archive passed validation and can proceed to the next step.')
else:
    print('Archive rejected. See the log for the validation reason.')

Archive passed validation and can proceed to the next step.


## 9. Summary

| Function | On success | On validation failure |
|---|---|---|
| `validate_tar_gz()` | Returns `True` | Raises an exception |
| `safe_validate_tar_gz()` | Returns `True` | Logs the exception and returns `False` |

**Note:** Catching `Exception` makes the wrapper broad: it also catches unexpected errors, not only `TarValidationError`. For stricter handling, catch `TarValidationError` specifically if that is the exception type used by the validation failures you want to suppress. Validation passing means the archive passed the configured FileAudit checks; it is not a guarantee against every possible security threat.

The temporary directory is automatically removed when the notebook kernel is restarted or when you run the cleanup cell below.

In [8]:
# Optional cleanup when you have finished the demo.
temp_dir.cleanup()
print('Temporary demo files cleaned up.')

Temporary demo files cleaned up.
